<a id="9"></a>
# 9. Advanced Python

Concepts that separate "knows Python syntax" from "writes idiomatic, efficient Python."


In [1]:
# =========================================================
# ITERATORS - objects that implement __iter__ and __next__
# (this is what actually powers "for" loops under the hood)
# =========================================================

class CountUpTo:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self            # an iterator returns itself from __iter__

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration   # signals "no more items" - this is how loops know to stop
        self.current += 1
        return self.current

for num in CountUpTo(5):
    print(num, end=" ")
print()

# Manually driving an iterator with next()
it = iter([10, 20, 30])
print(next(it), next(it), next(it))

1 2 3 4 5 
10 20 30


In [2]:
# =========================================================
# GENERATORS - a simpler way to write iterators, using "yield"
# Lazily produce values ONE AT A TIME instead of building a whole list in memory
# =========================================================

def count_up_to(limit):
    current = 1
    while current <= limit:
        yield current       # pauses here, returns a value, resumes on the next next() call
        current += 1

for num in count_up_to(5):
    print(num, end=" ")
print()

# Generator expression (already seen in section 4) - same idea, one-liner
squares_gen = (x**2 for x in range(5))
print(list(squares_gen))

# Real-world benefit: processing a huge file WITHOUT loading it all into memory
def read_large_file_lines(filepath):
    with open(filepath) as f:
        for line in f:
            yield line.strip()

1 2 3 4 5 
[0, 1, 4, 9, 16]


In [3]:
# =========================================================
# CLOSURES - a function that "remembers" variables from its enclosing scope
# =========================================================

def make_multiplier(factor):
    def multiplier(x):
        return x * factor    # 'factor' is remembered even after make_multiplier() has returned
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))   # 10 15 - each closure remembers its OWN 'factor'

10 15


In [4]:
# =========================================================
# DECORATORS - functions that wrap other functions to add behavior
# This is one of the most "advanced-feeling" but genuinely useful Python features.
# =========================================================
import functools
import time

def timer(func):
    @functools.wraps(func)          # preserves the original function's name/docstring
    def wrapper(*args, **kwargs):    # accepts ANY arguments, so it works on ANY function
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer                    # equivalent to: slow_function = timer(slow_function)
def slow_function():
    total = sum(range(1_000_000))
    return total

print(slow_function())

# Decorators with their OWN arguments require an extra layer of nesting
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(times=3)
def greet(name):
    print(f"Hello, {name}!")

greet("Alice")


slow_function took 0.017096s
499999500000
Hello, Alice!
Hello, Alice!
Hello, Alice!


In [5]:
# =========================================================
# CONTEXT MANAGERS - the "with" statement, and how to write your own
# =========================================================

class ManagedResource:
    def __enter__(self):
        print("Acquiring resource")
        return self          # this becomes the "as x" value

    def __exit__(self, exc_type, exc_value, traceback):
        print("Releasing resource")
        return False          # False = don't suppress exceptions; True = suppress them

with ManagedResource() as res:
    print("Using resource")

# Simpler way to write a context manager, using a generator + decorator
from contextlib import contextmanager

@contextmanager
def managed_resource():
    print("Acquiring resource (v2)")
    yield "the resource"       # code before yield = __enter__, code after = __exit__
    print("Releasing resource (v2)")

with managed_resource() as res:
    print(f"Using {res}")


Acquiring resource
Using resource
Releasing resource
Acquiring resource (v2)
Using the resource
Releasing resource (v2)


In [6]:
# =========================================================
# FUNCTIONAL PROGRAMMING TOOLS: map, filter, reduce
# =========================================================
from functools import reduce

nums = [1, 2, 3, 4, 5]

squared = list(map(lambda x: x**2, nums))          # apply a function to every element
evens = list(filter(lambda x: x % 2 == 0, nums))    # keep elements matching a condition
total = reduce(lambda acc, x: acc + x, nums)        # combine all elements into one value

print(squared)
print(evens)
print(total)

# In modern Python, comprehensions are usually preferred over map/filter for readability:
print([x**2 for x in nums])                # equivalent to the map() above
print([x for x in nums if x % 2 == 0])     # equivalent to the filter() above

[1, 4, 9, 16, 25]
[2, 4]
15
[1, 4, 9, 16, 25]
[2, 4]


In [7]:
# =========================================================
# *args / **kwargs UNPACKING (the reverse direction - "spreading" into a call)
# =========================================================

def add3(a, b, c):
    return a + b + c

values = [1, 2, 3]
print(add3(*values))          # unpacks the list into 3 positional args

kv = {"a": 1, "b": 2, "c": 3}
print(add3(**kv))             # unpacks the dict into keyword args

# Unpacking in assignment
first, *middle, last = [1, 2, 3, 4, 5]
print(first, middle, last)    # 1 [2, 3, 4] 5

# Merging collections via unpacking
combined_list = [*[1, 2], *[3, 4]]
combined_dict = {**{"a": 1}, **{"b": 2}}
print(combined_list, combined_dict)

6
6
1 [2, 3, 4] 5
[1, 2, 3, 4] {'a': 1, 'b': 2}


In [8]:
# =========================================================
# TYPE HINTS - optional, but standard in modern professional Python code
# (NOT enforced at runtime by default - they're for readability & tools like mypy/IDEs)
# =========================================================
from typing import List, Dict, Optional, Union

def greet(name: str) -> str:
    return f"Hello, {name}"

def process(items: List[int]) -> Dict[str, int]:
    return {"count": len(items), "sum": sum(items)}

def find_user(user_id: int) -> Optional[str]:
    # Optional[str] means "a str, or None"
    users = {1: "Alice"}
    return users.get(user_id)

def parse(value: Union[int, str]) -> int:
    # Union[int, str] means "either an int or a str"
    return int(value)

print(greet("Alice"))
print(process([1, 2, 3]))
print(find_user(2))


Hello, Alice
{'count': 3, 'sum': 6}
None


In [9]:
# =========================================================
# THE WALRUS OPERATOR := (Python 3.8+)
# Assigns a value AND returns it, in a single expression - reduces repeated calls.
# =========================================================

data = [1, 2, 3, 4, 5]

# Without walrus: computing len() twice
if len(data) > 3:
    print(f"List is long: {len(data)} items")

# With walrus: compute once, reuse the value
if (n := len(data)) > 3:
    print(f"List is long: {n} items")

# Very handy inside while loops reading input/data
values = iter([1, 2, 3, 0, 4])
while (val := next(values, None)) is not None:
    if val == 0:
        break
    print("got:", val)

List is long: 5 items
List is long: 5 items
got: 1
got: 2
got: 3


In [10]:
# =========================================================
# CONCURRENCY - a brief overview (each is a deep topic on its own)
# =========================================================
import threading
import time

def worker(name):
    time.sleep(0.1)
    print(f"Worker {name} done")

# threading -> good for I/O-bound tasks (network calls, file I/O) - NOT true parallelism
# because of the GIL (Global Interpreter Lock), which lets only one thread run
# Python bytecode at a time.
threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()          # wait for all threads to finish before continuing

# multiprocessing -> good for CPU-bound tasks, since each process gets its own
# Python interpreter and GIL, enabling true parallelism across CPU cores.
# import multiprocessing
# (not run here - process creation behaves differently inside notebooks)

# asyncio -> good for I/O-bound tasks using a SINGLE thread with cooperative
# multitasking (async/await). Modern standard for high-concurrency I/O (e.g. web servers).
import asyncio

async def async_worker(name):
    await asyncio.sleep(0.1)
    print(f"Async worker {name} done")

async def main():
    await asyncio.gather(*(async_worker(i) for i in range(3)))

await main()   # Jupyter notebooks support top-level await directly

Worker 1 doneWorker 0 done
Worker 2 done

Async worker 0 done
Async worker 1 done
Async worker 2 done
